In [89]:
import re #for regular expressions

def readInput(file_path):
    with open(file_path, 'r', encoding="utf-8") as f:
        input = f.read()

    plaintext = re.sub(r'[^A-Za-z]', '', input) #only keeps letters
    plaintext = plaintext.upper() #convert to uppercase

    return plaintext[:100000]

In [90]:
import random
import string

def generate_key():
    letters = list(string.ascii_uppercase)
    letters_randomisez = letters.copy()
    random.shuffle(letters_randomisez)
    return dict(zip(letters, letters_randomisez))

def encrypt(plaintext, key):
    ciphertext = ""
    for x in plaintext:
        ciphertext += key[x]
    return ciphertext

def decrypt(ciphertext, key):
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    decryption_mapping = dict(zip(key, alphabet))

    decrypted_message = ""
    for letter in ciphertext:
        decrypted_message += decryption_mapping[letter]

    return decrypted_message


In [91]:
def letters_frequency(ciphertext):
    freq = dict()
    for c in ciphertext:
        if c not in freq:
            freq[c] = 1
        else:
            freq[c] += 1
    return freq

def sort_letters_by_freq(ciphertext):
    letters_freq = letters_frequency(ciphertext)
    sorted_letters_freq = sorted(letters_freq.items(), key=lambda x: x[1], reverse = True)
    return sorted_letters_freq

In [92]:
def guess_initial_key(ciphertext):
    sorted_letters_freq = sort_letters_by_freq(ciphertext)

    print("\n\nFrequency analysis on the ciphertext:")
    print("Letter |   Count  | Percentage")
    total_freq = len(ciphertext)
    for letter, count in sorted_letters_freq:
        percentage = count / total_freq * 100
        print(f"  {letter}    |   {count}  | {percentage:.2f}%")
        
    english_freq = list("ETAOINSHRDLCUMWFGYPBVKJXQZ")
    guessed_mapping = dict()
    i = 0
    for letter, count in sorted_letters_freq:
        guessed_mapping[letter] = english_freq[i]
        i += 1

    if i <= 25:
        for letter in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
            if letter not in guessed_mapping:
                guessed_mapping[letter] = english_freq[i]
                i += 1
    
    #return the key in a string format
    key = ""
    for letter in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        currect_letter = [key for key, value  in guessed_mapping.items() if value == letter][0]
        key += currect_letter
    return key

In [112]:
import math

# Log-frequencies for bigrams (useful for calculating the score of a given text)
english_bigrams = {
    'TH': 2.71, 'HE': 2.33, 'IN': 2.03, 'ER': 1.78, 'AN': 1.61,
    'RE': 1.41, 'ON': 1.32, 'AT': 1.21, 'EN': 1.13, 'ND': 1.07,
    'TI': 0.99, 'ES': 0.95, 'OR': 0.94, 'TE': 0.94, 'OF': 0.92,
    'ED': 0.90, 'IS': 0.89, 'IT': 0.88, 'AL': 0.88, 'AR': 0.87,
    'ST': 0.86, 'TO': 0.86, 'NT': 0.85, 'NG': 0.83, 'SE': 0.83
}

english_trigrams = {
    'THE': 1.81, 'AND': 0.73, 'ING': 0.72, 'HER': 0.42, 'ERE': 0.31,
    'ENT': 0.28, 'THA': 0.27, 'NTH': 0.27, 'WAS': 0.26, 'ETH': 0.24,
    'FOR': 0.24, 'DTH': 0.21, 'HAT': 0.21, 'ION': 0.21, 'TIO': 0.20,
    'VER': 0.20, 'TER': 0.19, 'HES': 0.19, 'OFT': 0.19, 'ITH': 0.19
}

def score_text(text):
    # calculates the score of a given text based on its digrams
    score = 0.0
    text_len = len(text)

    # digram score
    for i in range(text_len - 1):
        pair = text[i:i+2]
        if pair in english_bigrams:
            score += english_bigrams[pair]

    # trigram score
    for i in range(text_len - 2):
        triplet = text[i:i+3]
        if triplet in english_trigrams:
            score += english_trigrams[triplet] * 2.0  # trigram weight
            
    return score

def swap_letters(key):
    # in 80% of the cases, I make a big swap and 20% a big swap
    if random.random() < 0.8:
        i = random.randint(0, 25)
        j = (i + random.choice([-2, -1, 1, 2])) % 26
    else:
        i, j = random.sample(range(26), 2)
    key_list = list(key)
    key_list[i], key_list[j] = key_list[j], key_list[i]
    return ''.join(key_list)


def similarity_percentage(s1, s2):
    if len(s1) != len(s2):
        raise ValueError("Strings must have the same length!")

    count = 0
    for i in range(len(s1)):
        if s1[i] == s2[i]:
            count += 1
    return (count / len(s1)) * 100


def determine_key(ciphertext, real_key, iterations = 10000, temp0 = 15.0, cooling = 0.999):
    best_key = guess_initial_key(ciphertext)
    best_plaintext = decrypt(ciphertext, best_key)
    best_score = score_text(best_plaintext)

    temperature = temp0

    print(f"\n\nThe initial key obtained from unigram analysis is: {best_key}")
    print(f"The initial score is: {best_score:.2f}")
    print(f"The accuracy percentage is: {similarity_percentage(best_key, real_key):.2f}%\n")

    print("Starting the Hill Climbing algorithm...")

    for i in range(iterations):
        # create a small variation by swapping 2 random letters
        new_key = swap_letters(best_key)

        # decrypt the text with the new key and evaluate it
        new_plaintext = decrypt(ciphertext, new_key)
        new_score = score_text(new_plaintext)

        # accept the new key if it has higher score (the new key is better)
        # we might accept it even if it's worse to escape a local maximum
        delta = new_score - best_score
        if new_score > best_score or random.random() < math.exp(delta / temperature):
            best_key = new_key
            best_plaintext = new_plaintext
            best_score = new_score

        temperature *= cooling

        # occasionally, we print the progress
        if i % 500 == 0 and i != 0:
            print(f"\nAt iteration {i}, the score is: {best_score:.2f}")
            print(f"The best key until now is: {best_key}")
            print(f"The accuracy percentage is: {similarity_percentage(best_key, real_key):.2f}% ")


    # returns the best results found
    print("\nHill climbing completed.")
    print(f"Best score: {best_score:.2f}")
    print(f"Best key:   {best_key}\n")

    return best_key, best_plaintext, best_score


In [113]:
def main():
    print("The randomly generated key is:")
    key = generate_key()
    key_string = ""
    for k, value in key.items():
        #print(f"{k} -> {value}")
        key_string += value
    print(key_string)

    plaintext = readInput("emma.txt")
    ciphertext = encrypt(plaintext, key)    

    best_key, best_plaintext, best_score = determine_key(ciphertext, key_string)

In [114]:
main()

The randomly generated key is:
IUVJMAQFOCWHGKERSPDTLYBNZX


The initial key obtained from unigram analysis is: IRGJMAVFKNWHZOEUCPDTLYBSQX
The initial score is: 38368.99
The accuracy percentage is: 57.69%

Starting the Hill Climbing algorithm...

At iteration 500, the score is: 40519.24
The best key until now is: IZWKMLGFOSAHUJEXRPDTVYBQCN
The accuracy percentage is: 42.31% 

At iteration 1000, the score is: 40788.91
The best key until now is: IUZOMLGFKVNHQJEAXPDTRYBCWS
The accuracy percentage is: 42.31% 

At iteration 1500, the score is: 40788.91
The best key until now is: INVOMLGFKXZHUJESCPDTAYBWQR
The accuracy percentage is: 42.31% 

At iteration 2000, the score is: 48201.60
The best key until now is: IXWQMLJFOSNHGKERUPDTVYBZCA
The accuracy percentage is: 53.85% 

At iteration 2500, the score is: 51134.90
The best key until now is: IRXJMLQFOVGHNKESUPDTZYBCWA
The accuracy percentage is: 53.85% 

At iteration 3000, the score is: 51134.90
The best key until now is: ISWJMLQFOZVHUKEAGPDTR